# Word2Vec Paper Analysis: Efficient Estimation of Word Representations in Vector Space

### Paper Title:Efficient Estimation of Word Representations in Vector Space
Authors: Tomas Mikolov, Kai Chen, Greg Corrado, Jeffrey Dean

## 1. Problem Statement

Most traditional natural language processing (NLP) methods treat each word as a separate, unique symbol—like giving every word its own ID number. This approach has several drawbacks:
- **Sparsity:** The way words are represented (as one-hot vectors) results in huge, mostly empty data structures, making computations inefficient.
- **No semantic relationships:** There’s no built-in way for the model to understand that some words are related in meaning; for example, "king" and "queen" are just as different as "king" and "banana" in these systems.
- **Data inefficiency:** Because the model can’t recognize similarities between words, it needs a lot more data to learn about language patterns.
- **Limited scalability:** Simple methods such as counting word combinations (N-grams), quickly reach their limits, even when using very large amounts of text, and can’t capture deeper relationships or scale well to bigger tasks. While N-gram models can be trained on trillions of words, they fail to capture semantic relationships. For tasks like machine translation or speech recognition where high-quality data is limited (millions to few billions of words), we need more sophisticated representations that can generalize better.

**The Vision:**<br>
The authors aim to create distributed word representations where:
- Similar words have similar vector representations.
- Semantic relationships can be captured through vector arithmetic.
- Models can be trained efficiently on billions of words.
- The representations enable linear algebraic operations like: `vector("King") - vector("Man") + vector("Woman") ≈ vector("Queen")`

## 2. Assumptions, Goals, and Research Questions

### Key Assumptions
1. **Distributional Hypothesis:** Words appearing in similar contexts have similar meanings.
2. **Linear Relationships:** Semantic and syntactic relationships can be captured as linear transformations in vector space.
3. **Scalability Trade-off:** Simpler models trained on more data can outperform complex models on less data.
4. **Context Window Sufficiency:** Local context (few words before/after) contains enough information for learning meaningful representations.

### Research Questions
1. Can we design neural architectures that eliminate computational bottlenecks while preserving representation quality?
2. How do different architectural choices (CBOW vs Skip-gram) affect the types of relationships learned?
3. What's the relationship between training data size, vector dimensionality, and representation quality?
4. Can vector arithmetic capture complex linguistic relationships?

### Goals:
- Propose CBOW and Skip-gram models.
- Train them on very large corpora using efficient optimization.
- Evaluate using a novel test set of word analogies (e.g., Paris:France :: Rome:Italy).

## 3. Mathematical Models and Theoretical Concepts

We want to learn **dense vector representations** (`D`-dimensional embeddings) for each word in a large vocabulary `V`, using an efficient model that can be trained on billions of words.


Training time complexity for any model is given by:
```
O = E × T × Q
```
Where:
- `E` = number of epochs
- `T` = total number of training words (tokens)
- `Q` = computational complexity **per training word**, **depends on the model architecture**

### 3.1 Feedforward NNLM (baseline)

$$
Q = N \times D + N \times D \times H + H \times V
$$

* $N$ = context size (window)
* $D$ = dimension of word vector
* $H$ = hidden units
* $V$ = vocabulary size

**Dominant term:** $H \times V$

### 3.2 Recurrent NNLM (baseline)

$$
Q = H \times H + H \times V
$$

* Avoids fixed context size (RNN uses full history)
* Dominant cost: $H \times H$

### Hierarchical Softmax Optimization
To address the H×V bottleneck, the authors use Huffman tree-based hierarchical softmax:
- Traditional softmax: O(V) operations
- Hierarchical softmax: O(log₂(V)) operations
- Huffman trees assign shorter codes to frequent words
- Achieves ~2× speedup for vocabulary of 1M words

## 4. New Model Architectures


We want to learn a **mathematical model** that represents each word as a **vector in ℝⁿ** such that:

* **Similar words** are **close together** in the vector space.
* These vectors can support operations like:
$$
\text{vec("king")} - \text{vec("man")} + \text{vec("woman")} \approx \text{vec("queen")}
$$

This is known as **distributional semantics**: “You shall know a word by the company it keeps.”

> Now we define some terms:

| Symbol                       | Meaning                                            |
| ---------------------------- | -------------------------------------------------- |
| $V$                          | Vocabulary size (e.g. 10⁵–10⁶)                     |
| $D$                          | Embedding dimension (e.g. 100–300)                 |
| $T$                          | Number of training tokens                          |
| $E$                          | Number of training epochs                          |
| $N$                          | Window size (number of context words on each side) |
| $C$                          | Max skip distance (for Skip-gram)                  |
| $w \in V$                    | A word                                             |
| $\vec{v}_w \in \mathbb{R}^D$ | Embedding of word $w$ (input vector)               |
| $\vec{u}_w \in \mathbb{R}^D$ | Output embedding of word $w$                       |

We want to **learn these vectors** from raw text.


### CBOW MODEL (Continuous Bag of Words)

#### Objective:

Given the **context** words $w_{t-n}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+n}$, **predict** the target (center) word $w_t$.

#### Step-by-step:

#### (a) Input: Context words

Let the context window be size $2n$, centered around word at position $t$. For example:

```text
the cat sat on the mat
         ↑
       center
```

If the center word is "sat", context might be \["the", "cat", "on", "the"].

#### (b) Represent each context word by its vector

Each word $w_i$ in context has an embedding $\vec{v}_{w_i} \in \mathbb{R}^D$.

#### (c) Compute the **mean** context vector:

$$
\vec{h} = \frac{1}{2n} \sum_{i=1}^{2n} \vec{v}_{w_i}
$$

This is a simple **bag-of-words** average — no word order is used.

#### (d) Score for each word $w \in V$:

$$
\text{score}(w) = \vec{u}_w^\top \vec{h}
$$

Here, $\vec{u}_w$ is the **output** embedding of word $w$, and the score is a dot product.

#### (e) Convert scores to probabilities using **softmax**:

$$
P(w_t \mid \text{context}) = \frac{e^{\vec{u}_{w_t}^\top \vec{h}}}{\sum_{w \in V} e^{\vec{u}_w^\top \vec{h}}}
$$

#### (f) Loss function (cross-entropy):

We minimize the **negative log likelihood** of the correct word:

$$
\mathcal{L}_{\text{CBOW}} = -\log P(w_t \mid \text{context})
$$


#### Computational Complexity

Now we will see how to calculate cost per training step:

$$
Q = N \times D + D \times \log_2(V)
$$

* $N \times D$: averaging context vectors
* $D \times \log_2(V)$: computing scores via **hierarchical softmax** (explained later)

So CBOW is **very fast**, especially with hierarchical softmax.


### SKIP-GRAM MODEL

#### Objective:

Given the **center word** $w_t$, predict **each context word** in a window.

> Opposite of CBOW.


### Step-by-step:

#### (a) Input: Center word $w_t$

It has input embedding $\vec{v}_{w_t} \in \mathbb{R}^D$

#### (b) Output: Predict multiple context words $w_{t-n}, \dots, w_{t+n}$

Each has output embedding $\vec{u}_w \in \mathbb{R}^D$

#### (c) For each context word $w_c$, compute:

$$
\text{score}(w_c) = \vec{u}_{w_c}^\top \vec{v}_{w_t}
$$

#### (d) Softmax:

$$
P(w_c \mid w_t) = \frac{e^{\vec{u}_{w_c}^\top \vec{v}_{w_t}}}{\sum_{w \in V} e^{\vec{u}_w^\top \vec{v}_{w_t}}}
$$

#### (e) Loss:

We sum over all context words:

$$
\mathcal{L}_{\text{SG}} = -\sum_{c \in \text{context}} \log P(w_c \mid w_t)
$$



#### Computational Complexity

$$
Q = C \times (D + D \cdot \log_2(V))
$$

* $C$: number of context words
* $D$: vector dimension
* $\log_2(V)$: via hierarchical softmax

#### FINAL OBJECTIVE FOR TRAINING

For both models, we use:
* **Stochastic Gradient Descent (SGD)**
* **Backpropagation** to update both $\vec{v}_w$ and $\vec{u}_w$
* **Loss = sum over all training positions** of the negative log-likelihood



## 5. Methodology Deep Dive

### Training Process
1. **Data preprocessing**: Restrict vocabulary to most frequent words
2. **Context extraction**: Sliding window over text corpus
3. **Stochastic gradient descent**: With linear learning rate decay
4. **Hierarchical softmax**: For efficient probability computation

### Architecture Comparison Strategy
The authors use a systematic approach:
1. **Controlled experiments**: Same data, same dimensionality across models
2. **Computational complexity analysis**: Focus on operations that scale with vocabulary size
3. **Quality metrics**: Comprehensive evaluation on syntactic/semantic tasks

### Distributed Training (DistBelief)
- **Asynchronous SGD**: Multiple model replicas with gradient synchronization
- **Adaptive learning rates**: Using Adagrad optimization
- **Massive scale**: 100+ replicas across data center machines


## 6. COMPARISON WITH BASELINE METHODS

| Model         | Semantic Accuracy | Syntactic Accuracy | Total   |
| ------------- | ----------------- | ------------------ | ------- |
| RNNLM         | 9%                | 36%                | 35%     |
| NNLM          | 23%               | 53%                | 47%     |
| **CBOW**      | 24%               | **64%**            | 61%     |
| **Skip-gram** | **55%**           | 59%                | **56%** |


> We can see that skip-gram dominates in **semantic** tasks, CBOW is slightly better on **syntactic**.



## 7. Experimental Setup and Evaluation

### Datasets
- **Google News corpus**: 6 billion tokens for large-scale experiments
- **LDC corpora**: 320M words for controlled comparisons
- **Vocabulary**: Restricted to 1M most frequent words

### Evaluation Framework

#### Semantic-Syntactic Word Relationship Test
- **Semantic questions**: 8,869 questions (5 types)
  - Capital cities: Athens:Greece :: Oslo:Norway
  - Currency: Angola:kwanza :: Iran:rial
  - City-state: Chicago:Illinois :: Stockton:California
  - Gender: brother:sister :: grandson:granddaughter
  
- **Syntactic questions**: 10,675 questions (9 types)
  - Comparative: great:greater :: tough:tougher
  - Superlative: easy:easiest :: lucky:luckiest
  - Present participle: think:thinking :: read:reading
  - Past tense: walking:walked :: swimming:swam
  - Plural: mouse:mice :: dollar:dollars

#### Evaluation Method
- **Vector arithmetic**: `vec(b) - vec(a) + vec(c) ≈ vec(d)`
- **Nearest neighbor search**: Find closest word to computed vector
- **Strict accuracy**: Only exact matches count (synonyms are errors)
- **Cosine similarity**: Distance metric for vector comparisons

### Additional Benchmarks
- **Microsoft Sentence Completion Challenge**: 1,040 sentences with missing words
- **MSR Word Relatedness Test**: Syntactic similarity benchmark



## 8. Results and Key Findings

### Model Performance Comparison

#### On Semantic-Syntactic Test Set
- **Skip-gram**: 55% semantic, 59% syntactic (best overall)
- **CBOW**: 24% semantic, 64% syntactic (best syntactic)
- **NNLM**: 23% semantic, 53% syntactic
- **RNNLM**: 9% semantic, 36% syntactic

#### Key Insights
1. **Skip-gram excels at semantics**: Better at capturing meaning relationships
2. **CBOW excels at syntax**: Better at grammatical relationships
3. **Both outperform traditional neural LMs**: Despite simpler architecture
4. **Computational efficiency**: Orders of magnitude faster training

### Scaling Analysis

#### Effect of Dimensionality and Data Size
- **Diminishing returns**: Both dimensions and data size show diminishing improvements alone
- **Joint scaling**: Must increase both dimensionality and data size together
- **Optimal trade-off**: Balance between computational cost and performance

#### Training Time Analysis
- **CBOW**: 1 day for 783M words (300 dimensions)
- **Skip-gram**: 3 days for 783M words (300 dimensions)  
- **Distributed training**: Can handle trillion-word corpora

### Large-Scale Results (DistBelief)
- **1000-dimensional vectors**: 66.1% semantic, 65.1% syntactic accuracy
- **6B word training**: Achieves state-of-the-art performance
- **Computational efficiency**: 2.5 days × 125 CPU cores for Skip-gram

## 9. Learned Relationships Examples

### Semantic Relationships
- **Geography**: France:Paris :: Italy:Rome :: Japan:Tokyo
- **Comparative**: big:bigger :: small:smaller :: cold:colder  
- **Professional**: Einstein:scientist :: Messi:midfielder :: Mozart:violinist
- **Political**: Sarkozy:France :: Berlusconi:Italy :: Merkel:Germany

### Syntactic Relationships
- **Corporate**: Microsoft:Windows :: Google:Android :: Apple:iPhone
- **Chemical**: copper:Cu :: zinc:Zn :: gold:Au
- **Cultural**: Japan:sushi :: Germany:bratwurst :: France:tapas

### Quality Assessment
- **Accuracy**: ~60% exact match on relationship questions
- **Improvement strategies**: Using multiple examples (10 vs 1) improves accuracy by ~10%
- **Limitations**: Morphology not captured, synonyms counted as errors

## 10. Major Contributions

### Theoretical Contributions
1. **Architectural Innovation**: Demonstrated that simpler models can outperform complex ones
2. **Computational Analysis**: Systematic framework for analyzing neural LM complexity
3. **Linear Algebra Insight**: Showed that linguistic relationships have linear structure in vector space

### Practical Contributions
1. **Efficiency Breakthrough**: Made large-scale word embeddings practically feasible
2. **Quality Improvement**: Significant gains in semantic and syntactic understanding
3. **Scalability**: Enabled training on billion-word corpora
4. **Accessibility**: Provided open-source implementation

### Impact on Field
1. **Paradigm shift**: From sparse to dense word representations
2. **Foundation for transfer learning**: Word embeddings became standard preprocessing
3. **Inspiration for future work**: Led to contextual embeddings (BERT, GPT)

## 11. IMPACT & LEGACY
This paper:
- Revolutionized NLP by enabling scalable word embeddings.
- Inspired follow-up work like GloVe, FastText, ELMo, BERT.
- Is still one of the most cited papers in modern AI research.
- Introduced the now-standard Word2Vec family.



## 12. Implementing `Word2Vec` From Scratch

### Setups and Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import os
import pickle

### Building Data-Preprocessing Class

In this class we are going to implement following steps:
- Initialize with config parameters
- Build vocabulary
- Calculate subsampling probabilities
- Create noise distribution
- Implement sentence processing

In [2]:
class Word2VecDataPreprocessor:
    def __init__(self, min_count=5, subsampling_thresh=1e-5):# Initialize with config parameters where min_count is the minimum frequency of words to be included in the vocabulary and subsampling_thresh is the threshold for subsampling
        self.min_count = min_count# Minimum frequency of words to be included in the vocabulary
        self.subsampling_thresh = subsampling_thresh# Threshold for subsampling
        self.word2idx = {}# Dictionary to map words to indices
        self.idx2word = {}# Dictionary to map indices to words
        self.vocab_size = 0# Size of the vocabulary
        self.word_counts = Counter()# Counter to keep track of word frequencies
        self.noise_dist=None# Placeholder for noise distribution
        self.total_words = 0# Total number of words in the corpus. here it is 0 because we haven't processed any text yet.
        
    def build_vocab(self, sentences):
        print("Building vocabulary by subsmapling words.")
        #initialising word counts
        for sentence in sentences:
            self.word_counts.update(sentence)# Updating the word counts with the words in the sentence
        
        #Fildering out the rare words that is words that occur less than min_count times
        filtered_words = [word for word, count in self.word_counts.items()
                          if count >= self.min_count]# here we are filtering out the words that occur less than min_count times
        
        #Creating mappings of words
        self.word2idx = {word: idx for idx, word in enumerate(filtered_words)}# Mapping words to indices by enumerating the filtered words
        self.idx2word = {idx : word for idx, word in enumerate(filtered_words)}# Mapping indices to words by enumerating the filtered words
        self.vocab_size = len(self.word2idx)# Size of the vocabulary is the length of the word2idx dictionary
        
        # Calculating subsampling probabilities
        self.total_words = sum(self.word_counts.values())# Total number of words is the sum of all word counts
        self.subsampling_probs = {}# Dictionary to store subsampling probabilities for each word
        for word, count in self.word_counts.items():# For each word and its count in the word counts
            if word in self.word2idx:
                freq = count / self.total_words # Frequency of the word in the corpus
                self.subsampling_probs[word] = max(
                    0, 1 - np.sqrt(self.subsampling_thresh / freq)) # Subsampling probability is calculated using the formula 1 - sqrt(threshold / frequency) and ensuring it is non-negative
        
        # Building noise distribution for negative sampling (P(w)^0.75)
        freqs = np.array([self.word_counts[word] for word in filtered_words]) ** 0.75 # Raising the word counts to the power of 0.75 to build the noise distribution so that more frequent words have a higher probability of being sampled
        self.noise_dist = freqs / freqs.sum() # Normalizing the noise distribution by dividing by the sum of frequencies to ensure it sums to 1
        
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Total words: {self.total_words}")
        
    def subsample_sentence(self, sentence):
        # Subsampling words in a sentence based on the calculated probabilities
        return [word for word in sentence
                if word in self.word2idx
                and np.random.rand() > self.subsampling_probs[word]] # If the word is in the vocabulary and a random number is greater than the subsampling probability keep the word otherwise discard it
    
    def sentences_to_indices(self, sentences, subsample=True): # Converting sentences to indices based on the vocabulary
        indexed_sentences = [] # List to store indexed sentences
        for sentence in sentences: # Iterating through each sentence
            if subsample:
                sentence = self.subsample_sentence(sentence) # Subsampling the sentence if subsample is True
            indexed_sentence = [ self.word2idx[word] for word in sentence
                                 if word in self.word2idx] # Converting words to indices if they are in the vocabulary by looking them up in the word2idx dictionary and then appending them to the indexed_sentence list
            if len(indexed_sentence)>1: # at least 2 words are required to form a valid sentence
                indexed_sentences.append(indexed_sentence) # Appending the indexed sentence to the list of indexed sentences
        return indexed_sentences # Returning the list of indexed sentences
    
    def get_negative_samples(self, target_idx, num_samples):
        #here we are generating negative samples using noise distribution
        return np.random.choice(
            self.vocab_size,
            size=num_samples,
            p=self.noise_dist,
            replace=False
        ) # Generating negative samples by randomly choosing indices from the vocabulary size based on the noise distribution and ensuring no replacement (i.e., no duplicate samples)

### Implementing Base Word2Vec Class

The base class handles shared functionality between CBOW and Skip-gram models like:
- Weight initialization
- Training loop
- Common math operations
- Model persistence

We are using Base Word2Vec as `parent class` that holds the blueprint for all Word2Vec variants.

In [3]:
class BaseWord2Vec:
    
    def __init__(self, vocab_size, embedding_dim, window_size=5, learning_rate=0.025, neg_samples=5, batch_size=128):
        self.vocab_size = vocab_size # Size of the vocabulary
        self.embedding_dim = embedding_dim # Dimension of the word embeddings
        self.window_size = window_size # Context window size
        self.learning_rate = learning_rate # Learning rate for the model
        self.neg_samples = neg_samples # Number of negative samples for training
        self.batch_size = batch_size # Batch size for training
        
        # Initializing weights for input and output layers using Xavier initialization
        """
        Xavier initialization is a method to initialize the weights of neural networks to ensure that the variance of the outputs is similar to the variance of the inputs, which helps in faster convergence during training. It is used to prevent vanishing or exploding gradients by scaling the weights based on the number of input and output units.
        To do xavier initialisation we use a uniform distribution where the weights are sampled from a range that is determined by the number of input and output units. 
        """
        limit = np.sqrt(6 / (vocab_size + embedding_dim)) # Calculating the limit for Xavier initialization where 6 is a constant that helps in scaling the weights
        self.W_in = np.random.uniform(-limit, limit, (vocab_size, embedding_dim)).astype(np.float32) # Input weights matrix initialized with random values from a uniform distribution where -limit and limit are the bounds of the distribution and the shape is (vocab_size, embedding_dim)
        self.W_out = np.random.uniform(-limit, limit, (embedding_dim, vocab_size)).astype(np.float32) # Output weights matrix initialized similarly with shape (embedding_dim, vocab_size)
        self.loss_history = [] # List to store the loss history during training
        self.best_loss = float('inf') # Initializing best loss to infinity to keep track of the best model during training
    
    def sigmoid(self, x):
        # Sigmoid activation function
        return 1 / (1 + np.exp(-x))
    
    def train_batch(self, batch):
        raise NotImplementedError("This method should be implemented in subclasses") # Placeholder for the training batch method to be implemented in subclasses
    
    def generate_batches(self, indexed_sentences):
        raise NotImplementedError("This method should be implemented in subclasses") # Also placeholder for the batch generation method to be implemented in subclasses
    
    def train(self, indexed_sentences, epochs=3, early_stopping=True, patience=3):
        print(f"Training for {epochs} epochs with batch size of {self.batch_size}.")
        best_weight = None # Placeholder for the best weights during training
        patience_counter = 0 # Counter for early stopping patience
        
        for epoch in range(epochs):
            total_loss = 0 # Resetting total loss for the epoch
            batch_count = 0 # Resetting batch count for the epoch
            
            #Creating batches
            batches = list(self.generate_batches(indexed_sentences)) # Generating batches from the indexed sentences
            np.random.shuffle(batches) # Shuffling the batches to ensure randomness with each epoch
            
            #processing batches with the tqdm for the progress bar
            for batch in tqdm(batches, desc=f"Epoch {epoch + 1}/{epochs}"): # Iterating through each batch with a progress bar
                batch_loss = self.train_batch(batch) # Training the batch and getting the loss
                total_loss += batch_loss # Accumulating the total loss for the epoch
                batch_count += 1 # Incrementing the batch count
                
            avg_loss = total_loss / batch_count # Calculating the average loss for the epoch
            self.loss_history.append(avg_loss) # Appending the average loss to the loss history
            
            self.learning_rate *= 0.98 # Decaying the learning rate by multiplying it by 0.98
            
            # early stopping check
            if avg_loss < self.best_loss:
                self.best_loss = avg_loss # Updating the best loss if the current average loss is lower
                best_weight = (self.W_in.copy(), self.W_out.copy())# Saving the best weights
                patience_counter = 0 # Resetting the patience counter
            else:
                patience_counter += 1 # Incrementing the patience counter if the average loss did not improve
                if early_stopping and patience_counter >= patience:
                    print(f"Early stopping occurred after {epoch + 1} epochs.")
                    self.W_in, self.W_out = best_weight # Restoring the best weights
                    break
            
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - LR: {self.learning_rate:.6f}") # Printing the average loss and learning rate for the epoch
    
    def get_word_embeddings(self, word_idx):
        return self.W_in[word_idx] # Returning the word embedding for the given word index by looking it up in the input weights matrix
    
    def save_model(self, file_path):
        # Saving the model weights and vocabulary mappings to a file
        model_data = {
            'W_in': self.W_in,
            'W_out': self.W_out,
            'vocab_size': self.vocab_size,
            'embedding_dim': self.embedding_dim,
            'word2idx': self.preprocessor.word2idx,
            'idx2word': self.preprocessor.idx2word
        }
        with open(file_path, 'wb') as f:
            pickle.dump(model_data, f)
        print(f"Model saved to {file_path}")
    
    @classmethod # Load the model from a file
    def load_model(cls, file_path):
        # Loading the model weights and vocabulary mappings from a file
        with open(file_path, 'rb') as f:
            model_data = pickle.load(f)# Reading the model data from the file using pickle

        model = cls(model_data['vocab_size'], model_data['embedding_dim'])# Creating an instance of the class with the loaded vocabulary size and embedding dimension
        model.W_in = model_data['W_in']# Loading the input weights from the model data
        model.W_out = model_data['W_out']# Loading the output weights from the model data
        model.preprocessor = type('', (), {})()  # Dummy preprocessor
        model.preprocessor.word2idx = model_data['word2idx']# Loading the word to index mapping from the model data
        model.preprocessor.idx2word = model_data['idx2word']# Loading the index to word mapping from the model data

        return model       

### Writing Skip-Gram Model
In this class we will implemenet:
- Inheritance from BaseWord2Vec.
- Implement batch generation.
- Implementation of forward and backward pass.

In [4]:
class SkipGramNeg(BaseWord2Vec):
    
    #Writing skip-gram model with negative sampling
    def __init__(self, preprocessor, **kwargs):# Initializing the SkipGramNeg model with a preprocessor and additional keyword arguments where **kwargs allows passing additional parameters to the parent class
        super().__init__(preprocessor.vocab_size, **kwargs) # Calling the parent class constructor
        self.preprocessor = preprocessor # Storing the preprocessor instance for vocabulary and subsampling information
    
    def generate_batches(self, indexed_sentences):
        batch = [] # List to store the current batch of training data
        
        for sentence in indexed_sentences: # Iterating through each indexed sentence
            for i, target_idx in enumerate(sentence):# For each word in the sentence, we treat it as the target word
                #makking dynamic window size
                window = np.random.randint(1, self.window_size + 1) # Randomly choosing a window size between 1 and the specified window size
                start = max(0, i - window) # Calculating the start index of the context window
                end = min(len(sentence), i + window + 1) # Calculating the end index of the context window
                
                for j in range(start, end):
                    if j ==i:
                        continue # Skipping the target word itself
                    context_idx = sentence[j] # Getting the context word
                    
                    #Adding negative samples
                    neg_samples = self.preprocessor.get_negative_samples(target_idx, self.neg_samples) # Getting negative samples for the target word
                    for neg_idx in neg_samples:
                        batch.append((target_idx, neg_idx, 0)) # Appending the negative sample with a label of 0 (negative)
                        
                    if len(batch) >= self.batch_size:
                        yield batch # Yielding the batch when it reaches the specified batch size
                        batch = [] # Resetting the batch for the next iteration
        
        if batch: # If there are remaining samples in the batch after processing all sentences
            yield batch # Yielding the last batch if it is not empty
    
    def train_batch(self, batch):# Training a batch of data of (target, context, label)
        batch_loss = 0 # Initializing the batch loss to 0
        target_indices = np.array([item[0] for item in batch]) # Extracting target indices from the batch
        context_indices = np.array([item[1] for item in batch]) # Extracting context indices from the batch
        labels = np.array([item[2] for item in batch]) # Extracting labels from the batch
        
        #getting the embeddings for target and context words
        target_embs = self.W_in[target_indices] # Getting the embeddings for the target words from the input weights matrix
        
        # Making forward pass
        scores = np.sum(target_embs * self.W_out[context_indices], axis=1) # Calculating scores by taking the dot product of target embeddings and output weights for context words
        probs = self.sigmoid(scores) # Applying the sigmoid function to get probabilities
        
        #Calculating loss
        batch_loss = -np.sum(labels * np.log(probs + 1e-7) + -np.sum((1-labels) * np.log(1 - probs + 1e-7))) # Calculating the loss using binary cross-entropy with a small epsilon to avoid log(0)
        
        #backward pass
        errors = labels - probs # Calculating the error between labels and predicted probabilities
        
        #updating output weights
        grad_out = np.outer(errors, target_embs) # Calculating the gradient for output weights using outer product(batch size, embedding_dim)
        for i, context_idx in enumerate(context_indices):
            self.W_out[:, context_idx] -= self.learning_rate * grad_out[i] # Updating the output weights for each context word by subtracting the gradient scaled by the learning rate
            
        #Updating input weights
        grad_in = np.zeros_like(self.W_in) # Initializing the gradient for input weights
        for i, target_idx in enumerate(target_indices):
            grad_in[target_idx] += errors[i] * self.W_out[:, context_indices[i]] # getting the gradient for input weights by multiplying the error with the corresponding output weights
            
        self.W_in[target_indices] -= self.learning_rate * grad_in[target_indices] # Updating the input weights for the target words by subtracting the gradient scaled by the learning rate
        
        return batch_loss# Returning the batch loss for the training step

### Implementing CBOW Model

Following are the steps we are going to implement:
- Inheritance from BaseWord2Vec.
- Implementing batch generation.
- Finally implementing forward/backward pass.

In [5]:
class CBOWNeg(BaseWord2Vec):
    def __init__(self, preprocessor, **kwargs):
        super().__init__(preprocessor.vocab_size, **kwargs)
        self.preprocessor = preprocessor
        
    def generate_batches(self, indexed_sentences): # Generating (context, target) batches for the CBOW model with negative sampling
        batch = [] # List to store the current batch of training data
        for sentence in indexed_sentences:
            for i, target_idx in enumerate(sentence):
                #building dynamic window size
                window = np.random.randint(1, self.window_size + 1) # Randomly choosing a window size between 1 and the specified window size
                start = max(0, i - window) # Calculating the start index of the context window
                end = min(len(sentence), i + window + 1) # Calculating the end index of the context window
                
                context_indices = [sentence[j] for j in range(start, end) if j != i] # Getting the context indices by excluding the target word index
                
                if not context_indices:
                    continue
                
                # adding positive samples
                batch.append((context_indices, target_idx, 1)) # Appending the context indices, target index, and label 1 (positive) to the batch
                
                #adding negative samples
                neg_target = self.preprocessor.get_negative_samples(target_idx, self.neg_samples) # Getting negative samples for the target word
                
                for neg_idx in neg_target:
                    batch.append((context_indices, neg_idx, 0)) # Appending the negative sample with a label of 0 (negative) to the batch
                
                if len(batch) >= self.batch_size:# If the batch size is reached
                    yield batch # Yielding the batch
                    batch = [] # Resetting the batch for the next iteration
        
        if batch:
            yield batch # Yielding the last batch if it is not empty
    
    
    def train_batch(self, batch): #training on a batch of (context, target, label)
        batch_loss = 0 # Initializing the batch loss to 0
        context_list = [item[0] for item in batch] # Extracting context lists from the batch
        target_indices = np.array([item[1] for item in batch]) # Extracting target indices from the batch
        labels = np.array([item[2] for item in batch]) # Extracting labels from the batch
        
        #Crating context vectors
        context_vecs = np.zeros((len(batch), self.embedding_dim), dtype=np.float32) # Initializing context vectors with zeros and shape (batch_size, embedding_dim)
        for i, context in enumerate(context_list): # Iterating through each context in the batch
            context_vecs[i] = np.mean(self.W_in[context], axis=0) # Calculating the mean of the input weights for the context words to create a context vector
        
        # Making forward pass
        scores = np.sum(context_vecs * self.W_out.T[target_indices], axis=1) # Calculating scores by taking the dot product of context vectors and transposed output weights for target words. Transposing the output weights allows us to align the dimensions correctly for the dot product operation.
        probs = self.sigmoid(scores) # Applying the sigmoid function to get probabilities
        
        # Calculating loss
        batch_loss = -np.sum(labels*np.log(probs + 1e-7) + -np.sum((2-labels) * np.log(1 - probs + 1e-7))) # Calculating the loss using binary cross-entropy with a small epsilon to avoid log(0)
        
        # Backward pass
        errors = labels - probs # Calculating the error between labels and predicted probabilities (batch_size,)
        
        # Update output weights
        grad_out = np.outer(errors, context_vecs)  # (batch_size, emb_dim)
        for i, target_idx in enumerate(target_indices): # Iterating through each target index in the batch
            self.W_out[:, target_idx] -= self.learning_rate * grad_out[i] # Updating the output weights for each target word by subtracting the gradient scaled by the learning rate

        # Update input weights
        grad_in = np.zeros_like(self.W_in) # Initializing the gradient for input weights
        for i, context in enumerate(context_list): # Iterating through each context in the batch
            context_grad = errors[i] * self.W_out[:, target_indices[i]] / len(context) # Calculating the gradient for input weights by multiplying the error with the corresponding output weights and dividing by the number of context words
            for ctx_idx in context:# Iterating through each context index in the context list
                grad_in[ctx_idx] += context_grad # Accumulating the gradient for each context word

        self.W_in -= self.learning_rate * grad_in # Updating the input weights by subtracting the gradient scaled by the learning rate

        return batch_loss

### Implementing Word2VecEvaluator Class
We are implementing the `Word2VecEvaluator` class to evaluate and visualize trained Word2Vec embeddings. It implements three key functionalities:
1. **Similarity Search**: Find semantically similar words.
2. **Word Analogies**: Solve "a is to b as c is to ?" problems.
3. **Embedding Visualization**: 2D PCA projections of word vectors.

In [ ]:
class Word2VecEvaluator:
    def __init__(self, model):
        self.model = model # Storing the trained Word2Vec model instance
        self.preprocessor = model.preprocessor # Storing the preprocessor instance for vocabulary and subsampling information
        
    def find_similar_words(self, word, top_k=5): # Finding semantically similar words to a given word
        """ Here we are going to find similar words using cosine similarity.
        Cosine similarity is a measure of similarity between two non-zero vectors
        of an inner product space that measures the cosine of the angle between them."""
        if word not in self.preprocessor.word2idx:
             print(f"Word '{word}' not in vocabulary") # If the word is not in the vocabulary, print a message and return an empty list
             return []
        word_idx = self.preprocessor.word2idx[word] # Getting the index of the word from the vocabulary
        word_embedding = self.model.get_word_embeddings(word_idx) # Getting the word embedding for the given word index
        
        #computing similarities
        all_embeddings = self.model.W_in # Getting all word embeddings from the input weights matrix
        similarities = cosine_similarity([word_embedding], all_embeddings)[0] # Calculating cosine similarity
        #getting top k similar words
        top_indices = np.argsort(similarities)[-top_k-1:-1][::-1] # Getting the indices of the top k similar words by sorting the similarities in descending order and excluding the word itself (hence -1)
        return [(self.preprocessor.idx2word[idx], similarities[idx]) for idx in top_indices] # Returning a list of tuples containing the similar words and their corresponding similarity scores
    
    def word_analogy(self, word_a, word_b,word_c,top_k=1):
        words = [word_a, word_b, word_c] # List of words for the analogy
        idxs = {} # Dictionary to store indices of the words
        for word in words:
            if word not in self.preprocessor.word2idx:
                print(f"Word '{word}' not in vocabulary")
                return [] # If any word is not in the vocabulary, print a message and return an empty list
            idxs[word] = self.preprocessor.word2idx[word] # Storing the index of each word in the dictionary
        
        #getting the embeddings for the words
        emb_a = self.model.get_word_embeddings(idxs[word_a]) # Getting the embedding for word_a
        emb_b = self.model.get_word_embeddings(idxs[word_b]) # Getting the embedding for word_b
        emb_c = self.model.get_word_embeddings(idxs[word_c]) # Getting the embedding for word_c
        
        # Calculating the analogy vector
        analogy_vec = emb_b - emb_a + emb_c # The analogy vector is calculated by subtracting the embedding of word_a from word_b and adding the embedding of word_c
        analogy_vec /= np.linalg.norm(analogy_vec) # Normalizing the analogy vector to unit length
        
        # Finding the most similar words to the analogy vector
        all_embeddings = self.model.W_in # Getting all word embeddings from the input weights matrix
        all_embeddings_norm = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True) # Normalizing all embeddings to unit length
        similarities = np.dot(all_embeddings_norm, analogy_vec) # Calculating cosine similarity between the analogy vector and all embeddings
        
        #gettng top k similar words
        top_indices = np.argsort(similarities)[-top_k:][::-1] # Getting the indices of the top k similar words by sorting the similarities in descending order
        # Returning the top k similar words along with their similarity scores
        return [(self.preprocessor.idx2word[idx], similarities[idx]) for idx in top_indices]
    
    def visualize_embeddings(self, words=None, max_words=50, figsize=(14, 10)):
        if words is None: # If no specific words are provided, use the first max_words words from the vocabulary
            words = list(self.preprocessor.idx2word.values())[:max_words] # Getting the first max_words words from the vocabulary
        else: # If specific words are provided, filter them to include only those in the vocabulary
            words = [word for word in words if word in self.preprocessor.word2idx]

        # Getting embeddings
        word_ids = [self.preprocessor.word2idx[word] for word in words]
        embeddings = np.array([self.model.get_word_embedding(idx) for idx in word_ids]) # Getting the embeddings for the specified words by looking them up in the input weights matrix

        # Applying PCA
        pca = PCA(n_components=2)
        embeddings_2d = pca.fit_transform(embeddings)


        plt.figure(figsize=figsize)
        scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.7)

        # Adding labels
        for i, word in enumerate(words):
            plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]),fontsize=9, alpha=0.8)

        plt.title('Word Embeddings Visualization using PCA', fontsize=16)
        plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
        plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
        plt.grid(alpha=0.2)
        plt.show()